# Início

In [ ]:
pip install torchmetrics

In [5]:
import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader, TensorDataset
import torchvision
from torchvision import datasets
from torchvision.transforms import ToTensor
import torch.optim as optim
from torchmetrics.functional.classification import multiclass_f1_score
from sklearn.metrics import f1_score

import plotly.express as px
import plotly.graph_objects as go
import numpy as np
import pandas as pd
import copy
from copy import deepcopy
from tqdm import tqdm
import time
import os
from scipy.signal import butter, sosfiltfilt
from scipy.interpolate import PchipInterpolator
from scipy.spatial import distance
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score

In [4]:
device = (
    "cuda"
    if torch.cuda.is_available()
    else "mps"
    if torch.backends.mps.is_available()
    else "cpu"
)
print(f"Using {device} device")

Using cpu device


# Preprocessamento PAMAP2

In [6]:
def scipy_interpolate_signal(series, method='pchip'):
    y = series.values
    x = np.arange(len(y))

    # Identifica índices válidos (não-NaN)
    valid_mask = ~np.isnan(y)

    # Se todos forem NaN ou houver apenas 1 ponto válido, retorna como está
    if valid_mask.sum() < 2:
        return series

    x_valid = x[valid_mask]
    y_valid = y[valid_mask]
    interpolator = PchipInterpolator(x_valid, y_valid, extrapolate=True)
    y_filled = interpolator(x)

    return pd.Series(y_filled, index=series.index)

In [ ]:
actisP = ['lying', 'sitting', 'standing', 'walking', 'running', 'cycling',
          'Nordic', 'watching TV', 'computer', 'car driving', 'ascending stairs', 'descending stairs',
          'vacuum', 'ironing', 'folding', 'house cleaning', 'soccer', 'rope jumping']
activity_id = [1, 2, 3, 4, 5, 6, 7, 9, 10, 11, 12, 13, 16, 17, 18, 19, 20, 24]

In [17]:
# 1. Definir os sufixos das 17 variáveis de cada sensor IMU
imu_features = [
    'temp',
    'acc16_x', 'acc16_y', 'acc16_z',  # Aceleração (escala ±16g)
    'acc6_x', 'acc6_y', 'acc6_z',     # Aceleração (escala ±6g)
    'gyro_x', 'gyro_y', 'gyro_z',     # Giroscópio
    'mag_x', 'mag_y', 'mag_z',        # Magnetômetro
    'orient_x', 'orient_y', 'orient_z', 'orient_w' # Orientação (Quatérnios)
]
columns = ['timestamp', 'activity_id', 'heart_rate']
for sensor_location in ['hand', 'chest', 'ankle']:
    for feature in imu_features:
        columns.append(f'IMU_{sensor_location}_{feature}')
acc_cols = [
    'IMU_ankle_acc16_x', 'IMU_ankle_acc16_y', 'IMU_ankle_acc16_z',
    'IMU_hand_acc16_x', 'IMU_hand_acc16_y', 'IMU_hand_acc16_z',
    'IMU_chest_acc16_x', 'IMU_chest_acc16_y', 'IMU_chest_acc16_z']

file_path = '/content/drive/MyDrive/Doutorado Unicamp/Projeto/Dataset/PAMAP2_Dataset/Protocol/subject103.dat'
df = pd.read_csv(file_path, sep=r'\s+', header=None, names=columns)
df['activity_block'] = (df['activity_id'] != df['activity_id'].shift()).cumsum()
for col in acc_cols:
    df[col] = df.groupby('activity_block')[col].transform(
        lambda group: scipy_interpolate_signal(group, method='pchip')
    )
df = df.drop(columns=['activity_block'])

In [ ]:
def preprocess_pamap2(data_dir, window_size=150, step_size=75, subjects=range(101, 110)):
    """
    Pré-processa o dataset PAMAP2 para aceleração triaxial.

    Parâmetros:
    - data_dir: Caminho para a pasta contendo os arquivos 'subject101.dat', etc.
    - window_size: Tamanho da janela em amostras (150 para 3s a 50 Hz).
    - step_size: Passo da janela deslizante (75 para 50% de sobreposição).
    - subjects: Lista de IDs dos voluntários (101 a 109).

    Retorna:
    - Xdata: Array numpy de dimensão (N, 150, 3)
    - ydata: Array numpy de dimensão (N, 3) -> [pos_sensor, id_atividade, usuario]
    """
    X_list = []
    y_list = []

    # Mapeamento das colunas dos acelerômetros de 16g no PAMAP2
    # Mão (Hand): colunas 4, 5, 6
    # Peito (Chest): colunas 21, 22, 23
    # Tornozelo (Ankle): colunas 38, 39, 40
    sensor_cols = {
        0: [4, 5, 6],   # 0 para mão
        1: [21, 22, 23], # 1 para peito
        2: [38, 39, 40]  # 2 para tornozelo
    }

    activity_col = 1 # Coluna contendo o ID da atividade

    for subj in subjects:
        file_path = os.path.join(data_dir, f'subject{subj}.dat')
        if not os.path.exists(file_path):
            print(f"Arquivo não encontrado: {file_path}. Pulando...")
            continue

        print(f"Processando voluntário {subj}...")

        # Carrega o arquivo usando espaço em branco como separador
        data = pd.read_csv(file_path, sep=r'\s+', header=None)
        activities = data.iloc[:, activity_col].values

        # Para cada posição de sensor
        for pos_id, cols in sensor_cols.items():
            acc_data = data.iloc[:, cols].values

            # 1. Tratar valores ausentes (NaN) com interpolação linear
            df_acc = pd.DataFrame(acc_data)
            df_acc = df_acc.interpolate(method='linear').bfill().ffill()
            acc_clean = df_acc.values

            # 2. Reamostragem de 100 Hz para 50 Hz (redução por fator de 2)
            # Pegando amostras a cada 2 linhas
            indices_resampled = np.arange(0, len(acc_clean), 2)
            acc_resampled = acc_clean[indices_resampled]
            act_resampled = activities[indices_resampled]

            # 3. Recorte em janelas deslizantes de 3 segundos (150 amostras)
            for i in range(0, len(acc_resampled) - window_size + 1, step_size):
                window_x = acc_resampled[i:i + window_size]
                window_act = act_resampled[i:i + window_size]

                # Identifica a atividade predominante na janela (Moda)
                unique_acts, counts = np.unique(window_act, return_counts=True)
                dominant_activity = unique_acts[np.argmax(counts)]

                # Opcional: Ignorar atividade 0 (transições/sem rótulo)
                if dominant_activity == 0:
                    continue

                X_list.append(window_x)
                y_list.append([pos_id, dominant_activity, subj])

    Xdata = np.array(X_list)
    ydata = np.array(y_list)

    return Xdata, ydata

# --- Exemplo de Execução ---
# Substitua pelo caminho correto da sua pasta com os dados do PAMAP2
DATA_DIR = './protocol/'

if __name__ == "__main__":
    Xdata, ydata = preprocess_pamap2(DATA_DIR)
    print("\nProcessamento concluído!")
    print(f"Dimensões de Xdata: {Xdata.shape}")  # Esperado: (N, 150, 3)
    print(f"Dimensões de ydata: {ydata.shape}")  # Esperado: (N, 3)